# FinSight RAG — Phase 2, Part 1: FinBERT Sentiment

**Purpose:** Score the Management Discussion & Analysis (MD&A) section of
the Visa 10-K using ProsusAI/finbert, a BERT model fine-tuned on 10,000+
financial sentences. Compare against GPT-4o zero-shot classification.

**Input:** Visa 10-K cached on Drive (from Phase 1 ingestor)  
**Output:** `DocumentSentiment` — label, avg_score, signed_score, per-sentence breakdown  
**Next notebook:** `correlate_final.ipynb` — sentiment vs price chart

---

### Why FinBERT over GPT-4o for sentiment?

| | FinBERT | GPT-4o zero-shot |
|--|---------|------------------|
| Cost per filing | Free (local CPU) | ~$0.05 |
| Speed | ~2 min for full MD&A | ~30 sec |
| Financial domain | Fine-tuned on financial text | General purpose |
| Calibration | Well-calibrated confidence scores | Overconfident |
| Batch processing | Yes — 32 sentences at once | Rate-limited |

FinBERT wins on cost and calibration. We compare both in Cell 9 to show the difference.

### The signed score
Every sentence gets: **Positive → +score**, **Negative → -score**, **Neutral → 0**  
Average across all MD&A sentences = `signed_score` for the whole filing.  
This scalar is what `correlate_final.ipynb` plots against next-day price movement.

## Cell 1 — Install dependencies

FinBERT requires `transformers` and `torch`. First run downloads ~440MB model — cached after that.

In [ ]:
# Install all dependencies in one shot — correct order matters
# torch must install BEFORE transformers so transformers finds it
!pip install torch==2.4.1+cpu -f https://download.pytorch.org/whl/torch_stable.html -q
!pip install transformers==4.44.2 -q
!pip install --upgrade \
  langchain-core langchain-openai langchain-chroma \
  langchain-classic langchain-text-splitters langchain-community \
  chromadb openai pdfplumber beautifulsoup4 tenacity pandas -q
print('✅ All dependencies installed')

ERROR: Could not find a version that satisfies the requirement torch==2.4.1+cpu (from versions: 2.2.0, 2.2.0+cpu, 2.2.0+cpu.cxx11.abi, 2.2.0+cu118, 2.2.0+cu121, 2.2.0+rocm5.6, 2.2.0+rocm5.7, 2.2.1, 2.2.1+cpu, 2.2.1+cpu.cxx11.abi, 2.2.1+cu118, 2.2.1+cu121, 2.2.1+rocm5.6, 2.2.1+rocm5.7, 2.2.2, 2.2.2+cpu, 2.2.2+cpu.cxx11.abi, 2.2.2+cu118, 2.2.2+cu121, 2.2.2+rocm5.6, 2.2.2+rocm5.7, 2.3.0, 2.3.0+cpu, 2.3.0+cpu.cxx11.abi, 2.3.0+cu118, 2.3.0+cu121, 2.3.0+rocm5.7, 2.3.0+rocm6.0, 2.3.1, 2.3.1+cpu, 2.3.1+cpu.cxx11.abi, 2.3.1+cu118, 2.3.1+cu121, 2.3.1+rocm5.7, 2.3.1+rocm6.0, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)
ERROR: No matching distribution found for torch==2.4.1+cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Cell 2 — Restart runtime

Clears stale package state. Colab will show 'Session crashed' — expected. Continue from Cell 3.

In [ ]:
import os
os.kill(os.getpid(), 9)

## Cell 3 — Imports

In [ ]:
import os, io, re, time, logging, requests, pdfplumber
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
from bs4 import BeautifulSoup

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from langchain_core.documents import Document

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print('✅ Imports done')

✅ Imports done


## Cell 4 — Mount Drive & set paths

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/finsight-rag-phase1'
CACHE_DIR  = f'{DRIVE_BASE}/data/filings'

# Load OpenAI key (needed for GPT-4o comparison in Cell 9)
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except Exception:
    os.environ['OPENAI_API_KEY'] = 'sk-...'  # paste key here if needed
    print('⚠️  Key set directly')

cached = list(Path(CACHE_DIR).glob('*.bin'))
if cached:
    for f in cached:
        print(f'✅ Found cached filing: {f.name} ({f.stat().st_size // 1000} KB)')
else:
    print('❌ No cached filings — run ingestor_final.ipynb first')

Mounted at /content/drive
✅ API key loaded from Colab Secrets
✅ Found cached filing: V_10-K_2025-11-06.bin (2909 KB)


## Cell 5 — Load the Visa filing from cache

Reconstructs the full filing text from the `.bin` cache.
No network calls — runs in ~5 seconds.

In [ ]:
def load_filing_text(cache_dir: str = CACHE_DIR, ticker: str = 'V') -> str:
    """
    Load the cached filing bytes and extract full plain text.
    Returns the complete document as a single string.
    """
    bins = list(Path(cache_dir).glob(f'{ticker}_*.bin'))
    if not bins:
        raise FileNotFoundError(f'No cached filing for {ticker} in {cache_dir}')

    # Use the most recent filing
    cache_path = sorted(bins)[-1]
    print(f'Loading: {cache_path.name} ({cache_path.stat().st_size // 1000} KB)')

    raw_bytes = cache_path.read_bytes()
    sample    = raw_bytes[:500].decode('utf-8', errors='ignore').lower()

    if any(m in sample for m in ['<html', '<!doctype', '<document']):
        soup = BeautifulSoup(raw_bytes, 'html.parser')
        for tag in soup(['script', 'style', 'head', 'nav', 'footer']):
            tag.decompose()
        text = re.sub(r'\n{3,}', '\n\n', soup.get_text(separator='\n')).strip()
    else:
        # PDF fallback
        with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
            text = '\n'.join(page.extract_text() or '' for page in pdf.pages)

    print(f'✅ Extracted {len(text):,} characters ({len(text.split()):,} words)')
    return text


filing_text = load_filing_text()
print(f'\nFirst 300 chars:\n{filing_text[:300]}')

Loading: V_10-K_2025-11-06.bin (2909 KB)
✅ Extracted 463,626 characters (67,072 words)

First 300 chars:
0001403161
2025
FY
FALSE
50
50
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#AccruedLiabilitiesCurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
h


## Cell 6 — Extract the MD&A section

The Management Discussion & Analysis (Item 7) is the section where executives
describe business performance, risks, and outlook in plain English.
This is the highest-signal section for sentiment analysis — more informative
than financial tables (which are factual) or legal boilerplate (which is neutral).

**Extraction strategy:** Search for the SEC-standard Item 7 header using regex,
then capture text until Item 7A (the next standard section). Falls back to
the first 8,000 characters if the header isn't found.

In [ ]:
def extract_mda(full_text: str, max_chars: int = 12000) -> str:
    """
    Extract the MD&A section (Item 7) from a 10-K filing.

    Strategy: find ALL occurrences of the MD&A header, skip the table
    of contents entry (always near the start), and use the second
    occurrence which is the actual section body.

    For Visa's 2025 HTML filing, the TOC entry appears at position ~35,402
    and the real MD&A content starts at position ~207,453.
    """
    pattern = re.compile(r'Management.{0,10}s\s+Discussion', re.IGNORECASE)
    matches = list(pattern.finditer(full_text))

    if not matches:
        logger.warning('MD&A header not found — using first %d chars', max_chars)
        return full_text[:max_chars]

    # Skip the first match if it's in the TOC (within first 50,000 chars)
    # The real MD&A content is always deep in the filing
    real_match = None
    for m in matches:
        if m.start() > 50_000:
            real_match = m
            break

    if not real_match:
        # Fallback: use last match if all are in first 50k (unlikely)
        real_match = matches[-1]

    start = real_match.start()
    print(f'  MD&A found at position {start:,} (skipped {len(matches)-1} TOC entries)')

    # Find Item 7A after this point to mark the end
    end_pattern = re.compile(r'ITEM\s+7A|Item\s+7A', re.IGNORECASE)
    end_match   = end_pattern.search(full_text, start)
    end_pos     = end_match.start() if end_match else start + max_chars

    mda = full_text[start:end_pos][:max_chars]
    print(f'✅ MD&A extracted: {len(mda):,} chars')
    print(f'\nFirst 400 chars of MD&A:\n{mda[:400]}')
    return mda


mda_text = extract_mda(filing_text)

  MD&A found at position 207,453 (skipped 3 TOC entries)
✅ MD&A extracted: 12,000 chars

First 400 chars of MD&A:
Management’s Discussion and Analysis of Financial Condition and Results of Operations
This management’s discussion and analysis provides a review of the results of operations, financial condition and liquidity and capital resources of Visa Inc. and its subsidiaries (Visa, we, us, our or the Company) on a historical basis and outlines the factors that have affected recent earnings, as well as those


## Cell 7 — Load FinBERT

`ProsusAI/finbert` is a BERT-base model fine-tuned on ~10,000 financial sentences
from analyst reports, earnings calls, and financial news.
It predicts **Positive / Negative / Neutral** with calibrated confidence scores.

**First run:** downloads ~440MB from HuggingFace. Subsequent runs load from cache instantly.

`@functools.lru_cache(maxsize=1)` ensures the model is loaded exactly once per session,
regardless of how many times `score_sentence()` is called.

In [ ]:
import functools

@functools.lru_cache(maxsize=1)
def _load_finbert_pipeline():
    """
    Load ProsusAI/finbert — cached so it loads only once per session.
    First call downloads ~440MB. Subsequent calls are instant.
    """
    from transformers import pipeline
    print('Loading ProsusAI/finbert (~440MB on first run)...')
    pipe = pipeline(
        task='text-classification',
        model='ProsusAI/finbert',
        tokenizer='ProsusAI/finbert',
        return_all_scores=True,  # return scores for all 3 classes
        device=-1,               # -1 = CPU; 0 = first GPU if available
    )
    print('✅ FinBERT loaded')
    return pipe


# Test with a known financial sentence
pipe = _load_finbert_pipeline()

test_sentences = [
    'Net revenues increased 10% driven by strong payments volume growth.',
    'We face significant regulatory uncertainty that could adversely affect our business.',
    'The following table presents our consolidated financial results.',
]

print('\n── FinBERT test on known sentences ─────────────────────────')
for sent in test_sentences:
    results = pipe(sent[:512], truncation=True)[0]
    best    = max(results, key=lambda x: x['score'])
    emoji   = {'positive': '🟢', 'negative': '🔴', 'neutral': '⚪'}.get(best['label'].lower(), '❓')
    print(f'{emoji} {best["label"].capitalize()} ({best["score"]*100:.0f}%) — {sent[:70]}')

Loading ProsusAI/finbert (~440MB on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


✅ FinBERT loaded

── FinBERT test on known sentences ─────────────────────────
🟢 Positive (96%) — Net revenues increased 10% driven by strong payments volume growth.
🔴 Negative (91%) — We face significant regulatory uncertainty that could adversely affect
⚪ Neutral (93%) — The following table presents our consolidated financial results.


## Cell 8 — Score the full MD&A section

Splits the MD&A into individual sentences, scores each with FinBERT in batches of 32,
and aggregates into a `DocumentSentiment` with:
- `label` — overall sentiment (most frequent across sentences)
- `avg_score` — average model confidence
- `signed_score` — mean of (+score, -score, 0) across all sentences → used in correlation chart
- `per_sentence` — full breakdown for inspection

In [ ]:
from dataclasses import dataclass

@dataclass
class SentenceResult:
    text:         str
    label:        str    # 'positive' | 'negative' | 'neutral'
    score:        float  # model confidence 0–1
    signed_score: float  # +score, -score, or 0

    @property
    def emoji(self) -> str:
        return {'positive': '🟢', 'negative': '🔴', 'neutral': '⚪'}.get(self.label, '❓')


@dataclass
class DocumentSentiment:
    label:          str
    avg_score:      float
    signed_score:   float  # key output for correlate.py
    sentence_count: int
    per_sentence:   list

    @property
    def summary(self) -> str:
        emoji = {'positive': '🟢', 'negative': '🔴', 'neutral': '⚪'}.get(self.label, '❓')
        return (
            f'{emoji} {self.label.capitalize()} · '
            f'{self.avg_score*100:.0f}% confidence · '
            f'{self.signed_score:+.3f} signed score · '
            f'{self.sentence_count} sentences'
        )


def score_document(text: str, batch_size: int = 32, min_len: int = 20) -> DocumentSentiment:
    """
    Score a full document section sentence by sentence using FinBERT.

    Args:
        text:       Document text (MD&A section).
        batch_size: Sentences per FinBERT batch (32 is optimal for CPU).
        min_len:    Minimum sentence length in chars — filters headers/labels.

    Returns:
        DocumentSentiment with aggregate and per-sentence scores.
    """
    pipe = _load_finbert_pipeline()

    # Split into sentences (simple regex — good enough for financial prose)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) >= min_len]
    print(f'Scoring {len(sentences)} sentences in batches of {batch_size}...')

    results: list[SentenceResult] = []

    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        # FinBERT max input length is 512 tokens — truncate long sentences
        batch_results = pipe(batch, truncation=True, max_length=512)

        for sent, result_list in zip(batch, batch_results):
            best  = max(result_list, key=lambda x: x['score'])
            label = best['label'].lower()
            score = best['score']
            signed = score if label == 'positive' else (-score if label == 'negative' else 0.0)

            results.append(SentenceResult(
                text=sent, label=label, score=score, signed_score=signed
            ))

        if (i // batch_size + 1) % 5 == 0:
            print(f'  ...{i + batch_size}/{len(sentences)} sentences scored')

    # Aggregate
    avg_score    = sum(r.score for r in results)        / len(results)
    signed_avg   = sum(r.signed_score for r in results) / len(results)
    label_counts = Counter(r.label for r in results)
    overall_label = label_counts.most_common(1)[0][0]

    return DocumentSentiment(
        label=overall_label,
        avg_score=avg_score,
        signed_score=signed_avg,
        sentence_count=len(results),
        per_sentence=results,
    )


print('Scoring Visa MD&A with FinBERT...\n')
sentiment = score_document(mda_text)

print(f'\n══ RESULT ══════════════════════════════════════════')
print(sentiment.summary)

Scoring Visa MD&A with FinBERT...

Scoring 86 sentences in batches of 32...

══ RESULT ══════════════════════════════════════════
⚪ Neutral · 86% confidence · +0.002 signed score · 86 sentences


## Cell 9 — Compare FinBERT vs GPT-4o zero-shot

Run both models on the same 10 sentences and compare labels and confidence.
This is the kind of evaluator insight that shows up in research papers and
is a strong talking point in technical interviews.

**What to look for:**
- FinBERT is better calibrated on financial jargon
- GPT-4o tends to be overconfident (scores closer to 1.0)
- Neither is perfect — disagreements are the most interesting cases

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Sample 10 sentences from across the MD&A
all_sentences = [r.text for r in sentiment.per_sentence]
step = max(1, len(all_sentences) // 10)
sample_sentences = all_sentences[::step][:10]

# GPT-4o zero-shot classifier
gpt_prompt = ChatPromptTemplate.from_messages([(
    'system',
    'You are a financial sentiment classifier. '
    'Classify the sentiment of financial text as exactly one of: positive, negative, neutral. '
    'Respond with ONLY the label and a confidence score 0–1. '
    'Format: LABEL|SCORE (e.g. positive|0.92)'
), (
    'human', '{sentence}'
)])
gpt_chain = gpt_prompt | ChatOpenAI(model='gpt-4o-mini', temperature=0) | StrOutputParser()

print('══ FinBERT vs GPT-4o-mini — 10 sample sentences ══════════\n')
print(f'{"Sentence":<55} {"FinBERT":<20} {"GPT-4o-mini":<20} {"Match"}')
print('─' * 105)

agree = 0
finbert_map = {r.text: r for r in sentiment.per_sentence}

for sent in sample_sentences:
    # FinBERT result
    fb_result = finbert_map.get(sent)
    if not fb_result:
        continue
    fb_label = fb_result.label
    fb_score = fb_result.score

    # GPT-4o-mini result
    try:
        raw = gpt_chain.invoke({'sentence': sent[:400]})
        parts = raw.strip().split('|')
        gpt_label = parts[0].lower().strip()
        gpt_score = float(parts[1]) if len(parts) > 1 else 0.0
    except Exception:
        gpt_label, gpt_score = 'error', 0.0

    match = '✅' if fb_label == gpt_label else '❌'
    if fb_label == gpt_label:
        agree += 1

    fb_str  = f'{fb_label} ({fb_score:.2f})'
    gpt_str = f'{gpt_label} ({gpt_score:.2f})'
    print(f'{sent[:54]:<55} {fb_str:<20} {gpt_str:<20} {match}')

print(f'\nAgreement rate: {agree}/{len(sample_sentences)} ({agree/len(sample_sentences)*100:.0f}%)')
print('\nNote: Disagreements are the most interesting cases — they reveal where')
print('financial domain fine-tuning (FinBERT) diverges from general language (GPT-4o-mini).')

══ FinBERT vs GPT-4o-mini — 10 sample sentences ══════════

Sentence                                                FinBERT              GPT-4o-mini          Match
─────────────────────────────────────────────────────────────────────────────────────────────────────────
Management’s Discussion and Analysis of Financial Cond  neutral (0.95)       neutral (0.85)       ✅
We are focused on extending, enhancing and investing i  positive (0.70)      positive (0.88)      ✅
Net revenue increased 11% over the prior year, primari  positive (0.96)      positive (0.85)      ✅
In August 2025, we released 
$1.4 billion
 of the as-c  neutral (0.93)       neutral (0.85)       ✅
During fiscal 2025, we recorded additional accruals of  positive (0.91)      neutral (0.75)       ❌
covered litigation have been reduced by 50% or more si  neutral (0.55)       neutral (0.75)       ✅
Non-GAAP financial results.                             neutral (0.80)       neutral (0.75)       ✅
Amortization of acquired intan

## Cell 10 — Per-sentence breakdown

Show the most positive and most negative sentences from the MD&A.
This is the kind of output you would show in a README GIF or demo.

In [ ]:
print('══ SENTIMENT DISTRIBUTION ══════════════════════════════════')
label_counts = Counter(r.label for r in sentiment.per_sentence)
total = len(sentiment.per_sentence)
for label, count in label_counts.most_common():
    pct   = count / total * 100
    bar   = '█' * int(pct / 2)
    emoji = {'positive': '🟢', 'negative': '🔴', 'neutral': '⚪'}.get(label, '❓')
    print(f'{emoji} {label.capitalize():<10} {bar:<25} {count:3d} sentences ({pct:.0f}%)')

print(f'\n══ MOST POSITIVE SENTENCES ══════════════════════════════════')
top_positive = sorted(
    [r for r in sentiment.per_sentence if r.label == 'positive'],
    key=lambda r: r.score, reverse=True
)[:3]
for i, r in enumerate(top_positive, 1):
    print(f'\n{i}. 🟢 Score: {r.score:.3f}')
    print(f'   {r.text[:200]}')

print(f'\n══ MOST NEGATIVE SENTENCES ══════════════════════════════════')
top_negative = sorted(
    [r for r in sentiment.per_sentence if r.label == 'negative'],
    key=lambda r: r.score, reverse=True
)[:3]
for i, r in enumerate(top_negative, 1):
    print(f'\n{i}. 🔴 Score: {r.score:.3f}')
    print(f'   {r.text[:200]}')

print(f'\n══ OVERALL MD&A SENTIMENT ═══════════════════════════════════')
print(sentiment.summary)

══ SENTIMENT DISTRIBUTION ══════════════════════════════════
⚪ Neutral    █████████████████████████████████████  65 sentences (76%)
🔴 Negative   ██████                     11 sentences (13%)
🟢 Positive   █████                      10 sentences (12%)

══ MOST POSITIVE SENTENCES ══════════════════════════════════

1. 🟢 Score: 0.959
   Net revenue increased 11% over the prior year, primarily due to the growth in processed transactions, nominal cross-border volume, and nominal payments volume, partially offset by higher client incent

2. 🟢 Score: 0.927
   GAAP operating expenses increased 30% over the prior year, primarily driven by higher litigation provision and personnel expenses.

3. 🟢 Score: 0.912
   During fiscal 2025, we recorded additional accruals of $2.2 billion to address claims associated with the interchange multidistrict litigation.

══ MOST NEGATIVE SENTENCES ══════════════════════════════════

1. 🔴 Score: 0.957
   During fiscal 2025 and 2024, we recorded charges within gene

## Cell 11 — Health check & save results

In [ ]:
import json

print('══ FINBERT HEALTH CHECK ═════════════════════════════════════')

has_sentences = sentiment.sentence_count > 10
has_signed    = abs(sentiment.signed_score) > 0
has_breakdown = len(sentiment.per_sentence) > 0
label_valid   = sentiment.label in ('positive', 'negative', 'neutral')

print(f'{"✅" if has_sentences else "❌"} Sentences scored: {sentiment.sentence_count}')
print(f'{"✅" if has_signed else "❌"} Signed score: {sentiment.signed_score:+.4f}')
print(f'{"✅" if label_valid else "❌"} Overall label: {sentiment.label}')
print(f'{"✅" if has_breakdown else "❌"} Per-sentence breakdown available')

# Save results to Drive for use in correlate_final.ipynb
results_path = Path(DRIVE_BASE) / 'data' / 'sentiment_results.json'
results_path.parent.mkdir(parents=True, exist_ok=True)

save_data = {
    'ticker':         'V',
    'filing_date':    '2025-11-06',
    'label':          sentiment.label,
    'avg_score':      round(sentiment.avg_score, 4),
    'signed_score':   round(sentiment.signed_score, 4),
    'sentence_count': sentiment.sentence_count,
    'label_breakdown': dict(Counter(r.label for r in sentiment.per_sentence)),
}
results_path.write_text(json.dumps(save_data, indent=2))
print(f'\n✅ Results saved to Drive: {results_path}')

print('\n══ RESULT ═══════════════════════════════════════════════════')
if all([has_sentences, has_signed, label_valid, has_breakdown]):
    print('🎉 FinBERT complete — open correlate_final.ipynb to continue')
    print(f'\nKey output for correlate.py:')
    print(f'  signed_score = {sentiment.signed_score:+.4f}')
    print(f'  (positive values → bullish tone, negative → bearish tone)')
else:
    print('⚠️  Some checks failed — review output above')

══ FINBERT HEALTH CHECK ═════════════════════════════════════
✅ Sentences scored: 86
✅ Signed score: +0.0024
✅ Overall label: neutral
✅ Per-sentence breakdown available

✅ Results saved to Drive: /content/drive/MyDrive/finsight-rag-phase1/data/sentiment_results.json

══ RESULT ═══════════════════════════════════════════════════
🎉 FinBERT complete — open correlate_final.ipynb to continue

Key output for correlate.py:
  signed_score = +0.0024
  (positive values → bullish tone, negative → bearish tone)
